In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.transforms import v2
from torch.utils.data.dataloader import DataLoader
from collections import OrderedDict

In [2]:
transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

batch_size = 2**1

trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=0)


testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=False, num_workers=0)

In [3]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

In [4]:
class MLP(nn.Module):
    def __init__(self, input_dim: int, mlp_depth: int, output_dim: int):
        super().__init__()

        self.network = nn.Sequential(OrderedDict({
            'lin_1': nn.Linear(in_features=input_dim, out_features=mlp_depth),
            'act_1': nn.GELU(),
            'lin_2': nn.Linear(in_features=mlp_depth, out_features=output_dim),
        }))
    def forward(self, x):
        return self.network(x)

In [5]:
import math

class Attention(nn.Module):
    def __init__(self, embed_dim: int, key_dim: int, value_dim: int):
        super().__init__()

        self.W_q = nn.Linear(in_features=embed_dim, out_features=key_dim, bias=False)
        self.W_k = nn.Linear(in_features=embed_dim, out_features=key_dim, bias=False)
        self.W_v = nn.Linear(in_features=embed_dim, out_features=value_dim, bias=False)

        self.key_dim = key_dim
    
    def forward(self, x):
        # x: [batch_size, num_tokens, embed_dim]
        # or: [num_tokens, embed_dim]

        Q = self.W_q(x)  # [..., num_tokens, key_dim]
        K = self.W_k(x)  # [..., num_tokens, key_dim]
        V = self.W_v(x)  # [..., num_tokens, embed_dim]

        scores = Q @ K.transpose(-2, -1)
        # scores: [..., num_tokens, num_tokens]

        scores = scores / math.sqrt(self.key_dim)

        attention_weights = F.softmax(scores, dim=-1)
        # softmax over "which key/token should this query attend to?"

        out = attention_weights @ V
        # out: [..., num_tokens, embed_dim]
        # matmul on tensors of dimension >= 2 batches on the last 2-dimensions to do the regular matrix multiplication there.

        return out

In [6]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim: int, keys_dim: list[int], value_dim: int):
        super().__init__()
        n_heads = len(keys_dim)

        self.heads = nn.ModuleList([
            Attention(
                embed_dim=embed_dim,
                key_dim=keys_dim[h],
                value_dim=value_dim,
            )
            for h in range(n_heads)
        ])
        
        self.aggregator = nn.Linear(
            in_features=n_heads * value_dim,
            out_features=embed_dim,
            bias=False,
            )
        
        self.keys_dim = keys_dim

    def forward(self, x):
        # x: [B, N, embed_dim]

        head_outputs = [head(x) for head in self.heads]
        # each: [B, N, value_dim]

        combined = torch.cat(head_outputs, dim=-1)
        # [B, N, n_heads * value_dim]

        return self.aggregator(combined)
        # [B, N, embed_dim]

In [7]:
class MNISTAttention(nn.Module):
    def __init__(self, embed_dim: int, key_dim: int, value_dim: int, n_heads: int, mlp_depth: int):
        super().__init__()
        self.MHA = MultiHeadAttention(
            embed_dim=embed_dim,
            keys_dim=[key_dim for _ in range(n_heads)],
            value_dim=value_dim,
        )
        self.decision = MLP(input_dim=embed_dim, mlp_depth=mlp_depth, output_dim=10)

    def forward(self, x):
        x = x.flatten(start_dim=1).unsqueeze(1)  # [B, 1, embed_dim]
        x = self.MHA(x)                          # [B, 1, embed_dim]
        x = x.squeeze(1)                         # [B, embed_dim]
        return self.decision(x)                  # [B, 10]

In [8]:
loss_fn = torch.nn.CrossEntropyLoss()
model: nn.Module = MNISTAttention(embed_dim=3*28*28, key_dim=8, value_dim=8, n_heads=4, mlp_depth=128).to(device=device)

In [9]:
muon_params = [p for p in model.parameters() if p.ndim == 2]
other_params = [p for p in model.parameters() if p.ndim != 2]

muon_opt = torch.optim.Muon(muon_params, lr=0.001, momentum=0.9)
other_opt = torch.optim.AdamW(other_params, lr=0.001, weight_decay=0.0)

In [10]:
for epoch in range(2):
    running_loss = 0.0

    for i, data in enumerate(trainloader, 0):
        inputs, labels = data
        inputs, labels = inputs.to(device=device), labels.to(device=device)

        if i % 2000 == 1:
            print(f"Step {i}, {running_loss / i}")
        
        muon_opt.zero_grad()
        other_opt.zero_grad()
        
        outputs = model(inputs)
        loss = loss_fn(outputs, labels)
        loss.backward()

        muon_opt.step()
        other_opt.step()

        running_loss += loss.item()

Step 1, 2.3578243255615234
Step 2001, 0.6670013658329745
Step 4001, 0.5133313774109657
Step 6001, 0.45171405519460617
Step 8001, 0.4103204971300662
Step 10001, 0.38554061244739024
Step 12001, 0.3668479448174036
Step 14001, 0.35216758472596
Step 16001, 0.3425932263239078
Step 18001, 0.33367796090491925
Step 20001, 0.3264145371090615
Step 22001, 0.31846787338986055
Step 24001, 0.3110425223976692
Step 26001, 0.3045484376365145
Step 28001, 0.30090189513365967
Step 1, 0.008805482648313046
Step 2001, 0.25155615486611443
Step 4001, 0.2448528246463109
Step 6001, 0.23699426731137704
Step 8001, 0.2387867812078649
Step 10001, 0.24041454904485482
Step 12001, 0.23925596575451064
Step 14001, 0.24096523946201917
Step 16001, 0.23912012548800535
Step 18001, 0.23545178777457684
Step 20001, 0.2362304471380368
Step 22001, 0.23391695972235949
Step 24001, 0.23205873301323166
Step 26001, 0.23292303334468348
Step 28001, 0.23378721538443217


In [11]:
correct = 0
total = 0

model.eval()
with torch.no_grad():
    for inputs, labels in testloader:
        inputs, labels = inputs.to(device=device), labels.to(device=device)
        outputs = model(inputs)
        preds = outputs.argmax(dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total
print(f"Accuracy: {accuracy:.3f}")

Accuracy: 0.937


## Training continued

In [13]:
for epoch in range(3):
    running_loss = 0.0

    for i, data in enumerate(trainloader, 0):
        inputs, labels = data
        inputs, labels = inputs.to(device=device), labels.to(device=device)

        if i % 2000 == 1:
            print(f"Step {i}, {running_loss / i}")
        
        muon_opt.zero_grad()
        other_opt.zero_grad()
        
        outputs = model(inputs)
        loss = loss_fn(outputs, labels)
        loss.backward()

        muon_opt.step()
        other_opt.step()

        running_loss += loss.item()

Step 1, 0.0017022110987454653
Step 2001, 0.19677546304382487
Step 4001, 0.22024006359270293
Step 6001, 0.21099717009718966
Step 8001, 0.2098524634668967
Step 10001, 0.21089138022487128
Step 12001, 0.21445836435591817
Step 14001, 0.21693525974401653
Step 16001, 0.21833583784297425
Step 18001, 0.21708995914735787
Step 20001, 0.21695558111907826
Step 22001, 0.2176689819990045
Step 24001, 0.21660787969846337
Step 26001, 0.21794433002025937
Step 28001, 0.21748248334895692
Step 1, 0.014777674339711666
Step 2001, 0.20779579481299765
Step 4001, 0.21677831256773827
Step 6001, 0.21212315114451877
Step 8001, 0.21090981499273637
Step 10001, 0.21133506467322277
Step 12001, 0.21403012668483293
Step 14001, 0.21562644233345546
Step 16001, 0.21640026477307522
Step 18001, 0.2168435930579249
Step 20001, 0.2183356836456561
Step 22001, 0.21756201572036749
Step 24001, 0.21716574669729832
Step 26001, 0.2181025322334341
Step 28001, 0.21790297620559784
Step 1, 0.05153029039502144
Step 2001, 0.21918634668880507

In [14]:
correct = 0
total = 0

model.eval()
with torch.no_grad():
    for inputs, labels in testloader:
        inputs, labels = inputs.to(device=device), labels.to(device=device)
        outputs = model(inputs)
        preds = outputs.argmax(dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total
print(f"Accuracy: {accuracy:.3f}")

Accuracy: 0.939


In [15]:
torch.save({"model": model.state_dict()}, "ckpt_3_mnsit_attention.pt")